# Probar un modelo de Hugging Face (demo simple)

Carga un modelo de Hugging Face, opcionalmente con un adaptador LoRA, y hazle preguntas en un bucle.

1. Elegir el modelo y (opcionalmente) el adaptador
2. Cargar modelo y tokenizador
3. Preguntar en directo

In [1]:
%pip install -q -U transformers peft accelerate


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Configuración

In [2]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_REPO = "Bellido/qwen2.5-0.5b-cafeterias-uam-lora" 
SYSTEM = "Eres un asistente de las cafeterías de la Universidad Autónoma de Madrid. Responde de forma breve y precisa."

In [3]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_REPO = "Bellido/qwen2.5-1.5b-cantoblanco-transporte-en-lora" 
SYSTEM = "You are an assistant that helps with train and bus transport information to Cantoblanco, Madrid, Spain, answer precisely and concisely."

## Cargar modelo

Si `ADAPTER_REPO` no es `None`, se aplica el adaptador LoRA encima del modelo base.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.float16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float32
else:
    DEVICE, DTYPE = "cpu", torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)

if ADAPTER_REPO is not None:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_REPO)
    print(f"Adaptador cargado desde {ADAPTER_REPO}")

model.eval()
print(f"Modelo {MODEL_ID} listo en {DEVICE}")

/Users/hugobellido/Desktop/Uni/GDGUAM/gdg_huggingface_2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:05<00:00, 57.08it/s]


Adaptador cargado desde Bellido/qwen2.5-1.5b-cantoblanco-transporte-en-lora
Modelo Qwen/Qwen2.5-1.5B-Instruct listo en mps


## 4. Función de preguntas

In [5]:
@torch.inference_mode()
def ask(model, question, max_new_tokens=96):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None, top_p=None, top_k=None,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## 5. Demo interactiva

Escribe preguntas. Deja la línea vacía para salir.

In [9]:
def demo():
    while True:
        q = input("Pregunta (vacío para salir): ").strip()
        if not q:
            break
        print(q,"->", ask(model, q), "\n")
demo()

What is the best way to go from Colmenar Viejo to Cantoblanco? -> The best option is usually the Cercanías C-4 from Colmenar Viejo (17 minutes). 

